# Notebook File Overview

In this notebook, we will be performing some statistical analysis common for clncal research:
1. Table one
    * chi-square
    * t-test
2. Logistic Regressions
    * univaraible
    * multivariable
3. Linear Regressions
    * univaraible
    * multivariable
4. Survival Analysis
    * Hazard Ratios
    * Kaplan-Meier Curves
    * Cox Regressions

# Add the code snippet to copy data *FROM* the *All of Us R and Cloud Storage snippets*

**Step 1: Run *Setup***

In [ ]:
library(tidyverse)  # Data wrangling packages.

**Step 2: Run the *copy_file_from_workspace_bucket.R* code snippet**

This will import:

* final_diabetes_ptsd.csv
* final_measurements.csv

as data frames

**NOTE: You will have to add another line of code to each step to get both files imported**

<div style=" background-color:#f8d7da; color:#842029; border:1px solid #f5c2c7; border-radius:8px; padding:20px; font-size:20px; "> 
<b>We are going to update the provided snippet, even past the <i>DON'T CHANGE FROM HERE</i> Warning</b><br> 
</div>

In [ ]:
# This snippet assumes that you run setup first

# This code copies a file from your Google Bucket into a dataframe

# replace 'test.csv' with the name of the file in your google bucket (don't delete the quotation marks)
name_of_file_in_bucket1 <- 'final_analysis.csv'
name_of_file_in_bucket2 <- 'condition_arranged.csv'
name_of_file_in_bucket3 <- 'visit_arranged.csv'

########################################################################
##
################# DON'T CHANGE FROM HERE ###############################
##
########################################################################

# Get the bucket name
my_bucket <- Sys.getenv('WORKSPACE_BUCKET')

# Copy the file from current workspace to the bucket
system(paste0("gsutil cp ", my_bucket, "/data/", name_of_file_in_bucket1, " ."), intern=T)
system(paste0("gsutil cp ", my_bucket, "/data/", name_of_file_in_bucket2, " ."), intern=T)
system(paste0("gsutil cp ", my_bucket, "/data/", name_of_file_in_bucket3, " ."), intern=T)

# Load the file into a dataframe
final_analysis  <- read_csv(name_of_file_in_bucket1)
head(final_analysis)

condition_arranged  <- read_csv(name_of_file_in_bucket2)
head(condition_arranged)

visit_arranged  <- read_csv(name_of_file_in_bucket3)
head(visit_arranged)

# Hands-On Guide: Logistic Regressions
Welcome!  
In this section we will   
1. Load the R packages we need,  
2. Prepare the data (convert characters → factors),  
3. Fit and interpret **univariable** logistic models, and  
4. Build a step-wise **multivariable** model, checking how odds ratios change.

## Load Required Packages
We rely on three tidyverse-friendly packages:  

* **dplyr** – data wrangling  
* **tidyr** – helper for reshaping (not strictly needed here, but nice to have)  
* **broom** – converts model outputs into tidy data frames for easy viewing / plotting  

Run the next code cell to attach them.

In [ ]:
library(dplyr)
library(tidyr)
library(broom)

## Convert Character Variables to Factors
Why?  
Logistic regression treats **numeric** predictors as continuous and **factor** predictors as categorical dummy variables.  
If we leave `gender`, `race`, or `ethnicity` as plain character strings, `glm()` will silently drop them or try to coerce them, which yields cryptic warnings.

Key points for students:  
* `factor()` turns text levels into proper categories.  
* `exclude = NULL` on `ethnicity` keeps missing values (`NA`) as an explicit “missing” level, which avoids losing rows later.
markdown

In [ ]:
fa <- final_analysis %>% 
  mutate(
    # categorical predictors
    gender    = factor(gender),
    race      = factor(race),
    ethnicity = factor(ethnicity, exclude = NULL),
    
    # diabetes binary
    diabetes_binary = if_else(diabetes == "Yes", 1L,
                         if_else(diabetes == "No", 0L, NA_integer_)),
    
    # PTSD binaries
    ptsd_doctor_binary = if_else(ptsd_doctor == "Yes", 1L,
                            if_else(ptsd_doctor == "No", 0L, NA_integer_)),
    ptsd_treatment_binary = if_else(ptsd_treatment == "Yes", 1L,
                               if_else(ptsd_treatment == "No", 0L, NA_integer_))
  )

## Inspect the Outcome Variable
Before modelling, always **look at your outcome**:  

1. Is it coded 0/1?  
2. Does it have enough 1’s to fit a model?  
3. Are there any NA’s that need attention?  

We’ll peek with `glimpse()` (structure) and `table()` (frequency counts).

In [ ]:
glimpse(fa$diabetes_binary)
table(fa$diabetes_binary, useNA = "ifany")

## Univariable (Crude) Logistic Models
Concept checkpoint for learners:  

* **Why use glm() funtion?**
  - glm stands for *generalized linear model* and in R, lets you model binary outcome correctly
  - Paired with *family = binomial* runs a logistic regression
* **Why one predictor at a time?**  
  – To see each variable’s *raw* association with the outcome, free of confounding.  
* **Why `family = binomial`?**  
  – Tells `glm()` to fit a logit link: log odds(Y = 1).  
* **How to read the output?**  
  – We exponentiate (via `broom::tidy(exponentiate = TRUE)`) so the coefficients are in **odds-ratio** form, which clinicians understand.

Run the code cell to fit five separate models (age, BMI, sex, PTSD doctor, PTSD treatment) and then view their ORs.

In [ ]:
m_uni_age <- glm(diabetes_binary ~ age,    data = fa, family = binomial)
m_uni_bmi <- glm(diabetes_binary ~ BMI,    data = fa, family = binomial)
m_uni_sex <- glm(diabetes_binary ~ gender, data = fa, family = binomial)
m_uni_doc <- glm(diabetes_binary ~ ptsd_doctor, data = fa, family = binomial)
m_uni_treat <- glm(diabetes_binary ~ ptsd_treatment, data = fa, family = binomial)

tidy(m_uni_age, exponentiate = TRUE, conf.int = TRUE)
tidy(m_uni_bmi, exponentiate = TRUE, conf.int = TRUE)
tidy(m_uni_sex, exponentiate = TRUE, conf.int = TRUE)
tidy(m_uni_doc, exponentiate = TRUE, conf.int = TRUE)
tidy(m_uni_treat, exponentiate = TRUE, conf.int = TRUE)

### Interpreting the Univariable Results

How would you interpret these results?


**Outcome:** `diabetes_binary` (1 = diabetes, 0 = no diabetes)  
**Model:** `glm(Y ~ X, family = binomial)` • Coefficients are shown as **odds ratios (OR)** with 95 % CIs.

| Predictor (coded as) | OR | 95 % CI | *p*-value | Interpretation |
|----------------------|----------------|----------------|-----------|-------------|
| **Age** (per year) | **1.06** | 1.04 – 1.09 | 1.1 × 10⁻⁹ | Each additional year of age is associated with a **6 % higher** odds of diabetes. |
| **BMI** (kg m⁻²) | **1.06** | 1.05 – 1.07 | 1.1 × 10⁻²¹ | Each 1-unit increase in BMI raises the odds of diabetes by **6 %**. |
| Male vs Female | 0.87 | 0.66 – 1.15 | 0.34 | No clear evidence that sex alone affects diabetes odds. |
| PTSD doctor (Yes vs No) | 1.03 | 0.81 – 1.33 | 0.81 | Consulting a doctor for PTSD shows **no significant unadjusted association** with diabetes. |
| **PTSD treatment (Yes vs No)** | **1.38** | 1.10 – 1.75 | 0.006 | Individuals who received PTSD treatment have **38 % higher** odds of diabetes (unadjusted). |


Limitations: these are unadjusted; relationships can weaken, strengthen, or reverse once we add covariates.

**Key take-aways**

1. Age and BMI are strong positive predictors of diabetes in univariable analyses.  
2. Receiving PTSD treatment is also positively associated with diabetes, but talking to a doctor about PTSD is not.  
3. Sex shows no statistically significant crude association.  
4. These are **unadjusted** models; effects may change when multiple predictors are entered together (see multivariable modelling in later steps).

> Next: build a multivariable model to assess whether these associations persist after adjusting for potential confounders (age, BMI, etc.).

##  Building a Multivariable Model – Step by Step
Teaching goal: show **how** adding blocks of predictors changes key effects.

Proposed sequence:  

| Step | Added variables | Rationale |
|------|-----------------|-----------|
| Base | Demographics (age, gender, race, ethnicity) | Standard confounder set |
| +BMI | Body mass index | Strong clinical risk factor |
| +BP + LDL | Systolic BP, LDL cholesterol | Cardiometabolic risk markers |
| Full | PTSD doctor, PTSD treatment | Variables of research interest |

We’ll build formulas incrementally with `update()`, fit four models, and store them in a named list for tidy comparison.

In [ ]:
## -----------------------------------------------------------
## 1.  Define and fit the *base* logistic-regression model
## -----------------------------------------------------------

form_base  <- diabetes_binary ~ age + gender + race + ethnicity
## Create a formula object:
# outcome  : diabetes_binary   (1 = diabetes, 0 = no diabetes)
# predictors: age (numeric), gender, race, ethnicity (categorical)
# The formula is stored in an object so we can extend it later.

m_base <- glm(form_base, data = fa, family = binomial)
## Fit the base model with glm():
# data = fa (prepared analysis data frame)
# family = binomial (means “logistic regression”)
# The fitted model object is saved as m_base.


## -----------------------------------------------------------
## 2.  Add blocks of predictors step by step
## -----------------------------------------------------------

# STEP 1 ─ add BMI
form_step1 <- update(form_base, . ~ . + BMI)
# update() takes an existing formula and:
#   . ~ . + BMI   → keep the left-hand side (outcome) and all current
#                   RHS terms, then append BMI.

# STEP 2 ─ add systolic blood pressure and LDL cholesterol
form_step2 <- update(form_step1, . ~ . + Systolic_BP + LDL_Cholesterol)
# Again, keep everything in form_step1 and add two more predictors.

# STEP 3 ─ add PTSD variables (doctor visit & treatment)
form_step3 <- update(form_step2, . ~ . +
                     ptsd_doctor_binary + ptsd_treatment_binary)
# This will become the “Full” model.


## -----------------------------------------------------------
## 3.  Fit each model and store them in a named list
## -----------------------------------------------------------

mods <- list(
  Base      = m_base,                                           # already fitted
  "+BMI"    = glm(form_step1, data = fa, family = binomial),    # Step-1 model
  "+BP+LDL" = glm(form_step2, data = fa, family = binomial),    # Step-2 model
  Full      = glm(form_step3, data = fa, family = binomial)     # Step-3 model
)
# • Each element is a glm object.
# • The names (“Base”, “+BMI”, etc.) act as labels when we iterate
#   over the list to compare odds ratios or goodness-of-fit.

### Use a helper to show the evolution of key odds ratios
*(a.k.a. the `purrr::imap_dfr()` trick)*  

#### Why wrap the workflow in a single helper pipe?
* **Concise:** one command builds the whole comparison table—no copy-and-paste of five separate `tidy()` calls.  
* **Error-proof:** you never forget to update a number when you rerun / refit a model.  
* **Focus:** notebook space is spent on *interpretation* instead of repetitive boilerplate code.

#### What is **purrr**?
`purrr` is the tidyverse package for **functional programming** in R.  
Think of it as a modern, pipe-friendly replacement for the base *apply* family.

| Key verb | What it does | Typical return |
|----------|--------------|----------------|
| `map()`      | loop over elements | list |
| `map_dfr()`  | loop & row-bind     | one data frame |
| `imap_*()`   | same as `map_*()` **plus** passes the element *name/index* as `.y` | varies |

---
#### Why is this better than a line-by-line approach?

| Task | Line-by-line | `purrr::imap_dfr()` helper |
|------|-------------|----------------------------|
| **Code length** | 4–6 nearly-identical calls + manual `bind_rows()` | Single 10-line pipe |
| **Maintenance** | Add/remove a model → edit many lines | Just tweak the `mods` list |
| **Consistency** | Easy to forget `exponentiate = TRUE` in one call | Option set once, applied everywhere |
| **Scalability** | Becomes unwieldy with 10+ models | Loop handles any length |
| **Pedagogical value** | Hides the iteration concept | Demonstrates *mapping* (functional programming) |

**Bottom line:**  
The helper

1. *Maps* over every model in `mods`,  
2. *Tidies* coefficients uniformly,  
3. *Filters* to the predictors of interest,  
4. *Tags* each row with the model name, and  
5. *Row-binds* everything into a clean, long-format table ready for display.

Students learn a reusable functional-programming pattern, and you avoid repetitive, error-prone code.

In [ ]:
## ------------------------------------------------------------------
## Summarise key predictors across the four models in `mods`
## ------------------------------------------------------------------
## Goal: produce one tidy data-frame showing odds ratios (OR) and
##       95 % CIs for BMI, BP, LDL-C and the two PTSD variables,
##       stacked by model (“Base”, “+BMI”, “+BP+LDL”, “Full”).
## ------------------------------------------------------------------

library(purrr)     # imap_dfr()
library(broom)     # tidy()
library(dplyr)     # %>%  filter()  mutate()  select()

purrr::imap_dfr(                   # iterate over list *mods*
  mods,                            # ← named list of glm objects
  ~ tidy(.x,                       # .x = each glm model
         exponentiate = TRUE,      # convert log-odds → odds ratios
         conf.int     = TRUE)      # add 95 % CI columns
      %>%
      filter(term %in%             # keep only the predictors we care about
               c("BMI",
                 "Systolic_BP",
                 "LDL_Cholesterol",
                 "ptsd_doctor_binary",
                 "ptsd_treatment_binary")) %>%
      mutate(model = .y,           # .y = name of the list element
             .before = 1)          # insert as first column
) %>%                              # after looping, we have a single data-frame
  select(model,                    # retain & order the columns we want
         term,
         estimate,                 # odds ratio
         conf.low,                 # lower 95 % CI
         conf.high,                # upper 95 % CI
         p.value)

### Evolution of the Multivariable Logistic Model
Below is a summary table (generated by the helper) showing how odds ratios for BMI, BP, LDL, and the PTSD variables change as we adjust for more covariates.

After running the code, discuss with the class:

1. **Confounding vs suppression** – Why might the PTSD doctor OR flip below 1 when we add BMI, BP, and LDL?  
2. **Clinical vs statistical significance** – LDL’s OR is statistically significant but the effect size is tiny.  
3. **Model building strategy** – Forward selection here is pedagogical; other options include backward elimination, LASSO, or theory-driven pre-specification.

#### Evolution of the Multivariable Logistic Model  
**Outcome:** `diabetes_binary` (1 = diabetes, 0 = no diabetes)  
All coefficients are reported as **odds ratios (OR)** with 95 % CIs.

| Model | Predictor | OR | 95 % CI | *p* |
|-------|-----------|----|---------|-----|
| **+BMI** | **BMI (kg m⁻²)** | **1.060** | 1.047 – 1.073 | 9 × 10⁻²¹ |
| **+BP + LDL** | BMI | 1.059 | 1.046 – 1.072 | 2 × 10⁻¹⁹ |
| | Systolic BP (mm Hg) | 1.002 | 0.994 – 1.011 | 0.60 |
| | **LDL-C (mg dL⁻¹)** | **0.996** | 0.993 – 0.999 | 0.010 |
| **Full** | BMI | 1.059 | 1.046 – 1.072 | 5 × 10⁻¹⁹ |
| | Systolic BP | 1.002 | 0.994 – 1.011 | 0.59 |
| | **LDL-C** | **0.996** | 0.993 – 0.999 | 0.015 |
| | **PTSD doctor (Yes)** | **0.69** | 0.48 – 0.99 | 0.043 |
| | **PTSD treatment (Yes)** | **1.61** | 1.16 – 2.25 | 0.005 |

> *Demographic covariates (age, gender, race, ethnicity)* are in every model but omitted from this table for brevity.

**Interpretation Highlights**
* **BMI** remains a strong, stable predictor throughout: each 1-unit increase raises diabetes odds by ≈ 6%, even after full adjustment.
* **LDL-cholesterol** shows a small but significant inverse association (≈ 0.4 % lower odds per mg dL⁻¹).  
  – Effect size is tiny; could reflect medication use or residual confounding.
* **Systolic BP** never reaches significance once BMI and LDL are included.
* **PTSD variables diverge** in the fully adjusted model:  
  • Talking to a doctor about PTSD is linked to **31 % lower** odds of diabetes (OR 0.69).  
  • Receiving PTSD treatment is linked to **61 % higher** odds (OR 1.61).  
  These opposing directions may capture differences in PTSD severity, health-care access, or unmeasured factors.
* Demographic controls (not shown) keep their earlier effects (age ↑, etc.).

**Teaching Points**
1. **Sequential modelling** helps students see how adding predictors can reveal, attenuate, or reverse associations.  
2. **Effect stability** (BMI) vs **effect emergence** (PTSD variables) provides a practical lesson on confounding and suppression.  
3. **Clinical relevance vs statistical significance**: tiny OR shifts (LDL) may be statistically real but clinically negligible.  
4. Encourage learners to **translate numbers into plain language**, e.g.:  
   “After accounting for demographics, BMI, blood pressure, lipids and PTSD factors, individuals who have undergone PTSD treatment have about 1.6-times the odds of diabetes compared with those who have not (95 % CI 1.16–2.25).”

**Students can now proceed to:**
* Inspect full coefficient tables (`tidy(mods$Full, exponentiate = TRUE)`),  
* Evaluate discrimination (ROC/AUC) and calibration,  
* Compare nested models formally (e.g., likelihood-ratio tests), or  
* Explore interactions (e.g., BMI × PTSD treatment).

Paste this block into a **Markdown cell** and keep each `|`-delimited row intact for proper table rendering in Jupyter.

# Hands-On Guide: Linear Regressions
In this section we will  
1. Examine the distribution of our continuous outcome (`Systolic_BP`),  
2. Fit and interpret a **simple (crude) age–BP model**, and  
3. Build a sequence of **multivariable** models to evaluate added predictors.

## Inspect the Outcome Distribution
*Always start with the data itself.*

Why a histogram?  
* Detect skew, outliers, ceiling/floor effects.  
* Decide if a log-transformation or robust method is warranted.  
* Provide students a visual sense of typical blood-pressure values.

> **Run the next code cell** to plot a 30-bin histogram of systolic BP.

In [ ]:
hist(fa$Systolic_BP, breaks = 30, main = "Systolic BP")

## Crude Linear Slope of Age on Systolic BP
**Purpose:** quantify the unadjusted relationship between age and systolic BP.

Key teaching points for the code that follows:  
1. `lm(Systolic_BP ~ age)` fits a straight-line model (β₀ + β₁·Age).  
2. `broom::tidy()` displays coefficients with 95 % CIs.  
3. Overlaying `abline(m_age)` on the scatterplot lets students *see* the model.

After running, interpret:  
* β₁ tells the expected change in BP per 1-year increase in age.  
* Check both **statistical** (p-value) and **clinical** relevance.

In [ ]:
m_age <- lm(Systolic_BP ~ age, data = fa)
tidy(m_age, conf.int = TRUE)

plot(Systolic_BP ~ age, data = fa)
abline(m_age, col = "red")

| Term              | Coefficient (β) | 95% CI            | t-stat | p-value        | Interpretation |
|------------------|-----------------|-------------------|--------|----------------|----------------|
| Intercept        | 110.1 mm Hg     | 105.4 – 114.7     | 46.4   | < 2 × 10⁻¹⁶    | Expected systolic BP at age 0 (a mathematical baseline, not clinically meaningful). |
| Age (per year)   | +0.29 mm Hg     | +0.18 – +0.40     | 5.04   | 5.2 × 10⁻⁷     | Each additional year of age is associated with an average 0.29 mm Hg increase in systolic BP. Over 10 years, that’s about +2.9 mm Hg. |

### Key takeaways

- **Statistical significance:** The age effect is highly significant (p < 0.000001), so we reject the null hypothesis of no age–BP relationship.  
- **Effect size:** The magnitude is modest but clinically plausible—aging ten years is linked to a ~3 mm Hg rise in systolic BP.  
- **Model fit:** A single-predictor model typically explains only a small portion of BP variability (expect R² ≈ 0.04). Check with `glance(m_age)`.  

### Next steps

- Add covariates (gender, BMI, LDL, diabetes) to examine confounding.  
- Inspect diagnostic plots (`plot(m_age)`) for linearity, homoscedasticity, and influential points.

## Building Multivariable Models Step-by-Step
Goal: show how adding covariates improves (or fails to improve) model fit.

| Step | Predictors added | Rationale |
|------|-----------------|-----------|
| **Demog** | age + gender + race + ethnicity | Adjust for basic demographics |
| **+ BMI + LDL** | add BMI and LDL-cholesterol | Capture adiposity & lipid effects |
| **+ Diabetes** | add diabetes status | Test incremental value of a clinical condition |

We use `update()` to extend the formula incrementally—students see **exactly** what changes each time.


In [ ]:
m0 <- lm(Systolic_BP ~ age + gender + race + ethnicity, data = fa)
m1 <- update(m0, . ~ . + BMI + LDL_Cholesterol)
m2 <- update(m1, . ~ . + diabetes_binary)

bind_rows(glance(m0), glance(m1), glance(m2)) %>%
  select(r.squared, adj.r.squared, AIC, BIC) %>%
  mutate(model = c("Demog", "+BMI+LDL", "+Diabetes"), .before = 1)

### Model-fit comparison for predicting **Systolic_BP**

* **R² / Adj. R²** – variance explained (penalised vs unpenalised).  
* **AIC / BIC** – information criteria; *lower is better* but BIC is harsher on extra terms.  

Encourage students to ask:  
* Does the new predictor meaningfully raise Adj. R²?  
* Does it lower AIC/BIC by > 2 units (rule-of-thumb for evidence)?

| Model            | Predictors added*                    | R²   | Adj. R² | AIC     | BIC     |
|------------------|--------------------------------------|------|---------|---------|---------|
| **Demog**        | age + gender + race + ethnicity      | 0.041 | 0.038   | 13 844  | 13 882  |
| **+ BMI + LDL**  | Demog + BMI + LDL-Cholesterol        | 0.103 | 0.100   | 13 734  | 13 783  |
| **+ Diabetes**   | + BMI + LDL + diabetes_binary        | 0.104 | 0.099   | 13 736  | 13 790  |

\* All models include the Demog block; each row shows what was added next.

#### How to read these numbers

1. **R-squared (R²)** – proportion of total variability in systolic BP explained by the model.  
2. **Adjusted R²** – R² penalised for extra predictors; use this to compare models with different numbers of variables.  
3. **AIC / BIC** – information criteria; *lower* values indicate a better trade-off between fit and complexity (BIC penalises extra terms more heavily than AIC).

#### Interpretation

* Going from the **Demog** model to **+ BMI + LDL** more than doubles the explained variance (R² rises from 4 % to ≈ 10 %).  
  • AIC drops by ~110 points and BIC by ~99 → strong evidence that adding BMI and LDL meaningfully improves model fit.

* Adding **diabetes status** on top of BMI and LDL (**+ Diabetes**) changes R² and adjusted R² by only ~0.0001 and actually nudges AIC **up** by 1.8 and BIC by 7.2.  
  • ΔAIC < 2 suggests the new term does **not** deliver a worthwhile improvement, and BIC clearly disfavors it.

Conclude:  
* **BMI + LDL** materially improve the model.  
* **Diabetes status** adds negligible explanatory value and worsens BIC → likely omit unless clinically essential.

#### Bottom line

The model with **age, sex, race, ethnicity, BMI, and LDL-cholesterol** offers the best balance of explanatory power and parsimony.  
Including **diabetes_binary** adds negligible information and is not justified by AIC/BIC; it can be omitted unless clinical reasoning dictates otherwise.

# Hands-On Guide: Survival Analysis
In this section we will  
1. **Wrangle** raw condition & visit tables into a *time-to-event* data set  
2. **Visualise** non-parametric *Kaplan–Meier* curves  
3. **Model** hazards with a *Cox proportional-hazards* regression  

## Build the Analysis Data Set  
*(from raw conditions to index & outcome dates)*  

This code block carries out four key tasks:

| Step | Action | Purpose for survival analysis |
|------|--------|--------------------------------|
| **1** | Label each row as **diabetes** or **CKD**; extract CKD stage into `ckd_severity`. | Defines exposure (diabetes) and event (CKD + severity). |
| **2** | Keep the **earliest** date per person per condition. | Ensures follow-up starts at first diabetes diagnosis. |
| **3** | `pivot_wider()` → create separate columns for `index_date` (diabetes) and `outcome_date` (CKD). | Simplifies later date arithmetic. |
| **4** | Filter to rows that **have diabetes** (`index_date` not NA). | Establishes the risk set. |

> *Teaching tip:* Emphasise that every survival analysis begins with a **“one-row-per-patient”** table containing  
> ‑ *start date* (time 0)  
> ‑ *end date* (event or censoring)  
> ‑ *status* (1 = event, 0 = censored)

In [ ]:
library(dplyr)
library(tidyr)
library(stringr)

# 1. Create 'condition' labels AND 'ckd_severity'
condition_cleaned <- condition_arranged %>%
  mutate(
    # Create the condition label
    condition = case_when(
      str_detect(str_to_lower(standard_concept_name), "diabetes") ~ "diabetes",
      str_detect(str_to_lower(standard_concept_name), "renal|kidney") ~ "ckd",
      TRUE ~ NA_character_
    ),
    # Map the severity (only for CKD rows)
    ckd_severity = case_when(
      condition == "ckd" & str_detect(str_to_lower(standard_concept_name), "stage 5|end stage|end-stage|dialysis|failure") ~ 5,
      condition == "ckd" & str_detect(str_to_lower(standard_concept_name), "stage 4") ~ 4,
      condition == "ckd" & str_detect(str_to_lower(standard_concept_name), "stage 3") ~ 3,
      condition == "ckd" & str_detect(str_to_lower(standard_concept_name), "stage 2") ~ 2,
      condition == "ckd" & str_detect(str_to_lower(standard_concept_name), "stage 1") ~ 1,
      TRUE ~ NA_real_
    )
  ) %>%
  filter(!is.na(condition))

# 2. Keep only the earliest date for each person per condition
condition_earliest <- condition_cleaned %>%
  group_by(person_id, condition) %>%
  slice_min(condition_start_datetime, with_ties = FALSE) %>%
  ungroup()

# 3. Pivot the data to create columns for index_date, outcome_date, and severity
final_surv <- condition_earliest %>%
  select(person_id, condition, condition_start_datetime, ckd_severity) %>%
  pivot_wider(
    names_from = condition,
    # We pivot both the date and the severity
    values_from = c(condition_start_datetime, ckd_severity)
  ) %>%
  # Rename and clean up the pivoted columns
  rename(
    index_date = condition_start_datetime_diabetes,
    outcome_date = condition_start_datetime_ckd,
    severity_at_outcome = ckd_severity_ckd
  ) %>%
  # Remove the helper column created for diabetes severity (which is always NA)
  select(-ckd_severity_diabetes)

# 4. Filter for index patients and handle ordering
final_surv <- final_surv %>%
  filter(!is.na(index_date))

## 5.2  Basic Censoring & Time-to-Event Calculation  

This next Code block adds fundamental censoring logic:

1. **Remove** CKD dates that *precede* diabetes (pre-existing disease).  
2. Assign a fixed **study end date** to patients who never develop CKD.  
3. Compute `time_to_event` in **days** and drop zero/negative durations.

Explain to students:  
* Censoring captures “observed until … then follow-up ended without event.”  
* Negative or zero times indicate data problems (e.g., swapped dates).


In [ ]:
# 1. Drop rows where CKD happened BEFORE Diabetes
# (These are pre-existing conditions, not incident events)
final_surv <- final_surv %>%
  filter(is.na(outcome_date) | outcome_date >= index_date)

# 2. Define a "Study End Date" for those who never got CKD
# In All of Us, this is often the date of the last data refresh or the patient's last visit
# For training simplicity, let's assume a fixed date (e.g., '2024-01-01') 
# or use a column you've pulled like 'last_visit_date'
study_end_date <- as.Date("2024-01-01")

# 3. Create the 'status' and 'time_to_event' columns
final_surv <- final_surv %>%
  mutate(
    # Status: 1 if they have CKD, 0 if they don't
    status = if_else(!is.na(outcome_date), 1, 0),
    
    # End Point: If they have CKD, use that date. If not, use the study end date.
    calc_end_date = if_else(status == 1, as.Date(outcome_date), study_end_date),
    
    # Calculate Time (usually in days or years)
    # difftime results in a 'difftime' object, so we wrap it in as.numeric
    time_to_event = as.numeric(difftime(calc_end_date, index_date, units = "days"))
  ) %>%
  # Remove people with 0 or negative time (sanity check)
  filter(time_to_event > 0)

## Refine Censoring with Visit Data  

This next code block replaces the crude fixed cut-off with each patient’s **last visit date**:

* Join the latest `visit_end_datetime` per `person_id`.  
* Recalculate `time_to_event` using `calc_end_date = outcome_date OR last_visit_date`.  
* Drop rows still lacking follow-up evidence.

Discuss: contrast **administrative censoring** (study end) vs **loss-to-follow-up** (no visits, no event).


In [ ]:
library(dplyr)

# 1. Clean the visit data: Keep only the most recent date per person
# We convert PERSON_ID to lowercase to match your other dataframe
visit_latest <- visit_arranged %>%
  rename(person_id = PERSON_ID) %>%
  group_by(person_id) %>%
  summarize(last_visit_date = max(visit_end_datetime, na.rm = TRUE)) %>%
  ungroup()

# 2. Join to your final analysis dataframe
final_surv <- final_surv %>%
  left_join(visit_latest, by = "person_id")

# 3. Update the survival logic
# If they have an outcome (CKD), use that. 
# If they don't (NA), use their last known visit.
final_surv <- final_surv %>%
  mutate(
    # Ensure dates are in Date format for subtraction
    index_date = as.Date(index_date),
    outcome_date = as.Date(outcome_date),
    last_visit_date = as.Date(last_visit_date),
    
    # Calculate the end point for the calculation
    calc_end_date = if_else(!is.na(outcome_date), outcome_date, last_visit_date),
    
    # Calculate the time column (Duration)
    time_to_event = as.numeric(difftime(calc_end_date, index_date, units = "days"))
  ) %>%
  # Drop anyone where we still have an NA for time (no visits and no outcome)
  # or where the time is 0 or negative
  filter(!is.na(time_to_event) & time_to_event > 0)

dim(final_surv)
head(final_surv, 5)

## Create the `Surv()` Object  

Convert two columns into a survival object:

In [ ]:
#install.packages("survival")
library(survival)
#install.packages("survminer")
library(survminer)

# Ensure time is numeric and status is 0/1
# We use the 'time_to_event' and 'status' columns created in the previous steps
final_surv <- final_surv %>%
  mutate(
    status = as.numeric(status),
    time_to_years = time_to_event / 365.25  # Optional: convert days to years for easier interpretation
  ) %>%
  mutate(severity_at_outcome = ifelse(is.na(severity_at_outcome), 0, severity_at_outcome))

# Create the Survival Object
surv_obj <- Surv(time = final_surv$time_to_years, event = final_surv$status)

## Kaplan–Meier Curves

Teaching points:

* **Step curve** shows probability of remaining CKD-free over time.
* **Confidence band:** non-parametric 95 % CI.
* **Risk table** supports interpretation (how many still at risk).
* **Log-rank p-value** tests equality of survival curves across severity levels.

plain-language description:
“Patients with stage 5 reach 50 % CKD within ~1.2 years, whereas stage 0 remains >95 % CKD-free at 5 years,” etc.

In [ ]:
# Make the in-notebook plotting window larger
options(repr.plot.width  = 20,   # inches
        repr.plot.height = 12)   # a bit taller for the risk table

library(survival)
library(survminer)

# Kaplan–Meier fit
km_fit <- survfit(surv_obj ~ severity_at_outcome, data = final_surv)

# ggsurvplot object (curve + risk table)
surv_plot <- ggsurvplot(
  km_fit,
  data             = final_surv,
  risk.table       = TRUE,
  risk.table.height = 0.30,      
  pval             = TRUE,
  conf.int         = TRUE,
  palette          = "jco",
  ggtheme          = theme_bw(base_size = 13),
  legend           = "right",
  title            = "Time from Diabetes Diagnosis to CKD by Severity Stage",
  xlab             = "Years since Diagnosis",
  ylab             = "Probability of remaining CKD-free"
)

surv_plot          # displays in notebook

## Cox Proportional-Hazards Model 
**Outcome:** Time from diabetes diagnosis to chronic kidney disease (CKD) onset  
**Predictor:** Baseline CKD-risk severity (`severity_at_outcome`; reference = severity 0)

1. **Hazard ratio (HR):** multiplicative change in instantaneous CKD risk vs. reference (severity 0).
2. **β (log-HR):** coefficient on log scale.
3. **Global tests:** LR, Wald, Score — p < 0.05 ⇒ at least one severity level differs.
4. **Concordance (C-index):** model’s ability to rank patients by risk (≥0.70 = good).

In [ ]:
# Run the Cox Regression
# This tells us the risk of CKD based on the severity stage
cox_model <- coxph(surv_obj ~ severity_at_outcome, data = final_surv)

# View the results
summary(cox_model)

### Model Summary
* *n* = 7 334 participants  
* Events (CKD) = 578  
* **Concordance (C-index)** = 0.752 ± 0.012  
* **Global tests**  
  * Likelihood-ratio χ² = 1223 (df = 5), *p* < 2 × 10⁻¹⁶  
  * Wald χ² = 1577 (df = 5), *p* < 2 × 10⁻¹⁶  
  * Score (log-rank) χ² = 3678 (df = 5), *p* < 2 × 10⁻¹⁶  

### Hazard Ratios by Severity Category

| Severity | β (log-HR) | HR (eᵝ) | 95 % CI | *p*-value | Interpretation |
|----------|-----------:|--------:|---------|-----------|----------------|
| **1** | **+3.27** | **26.38** | 19.28 – 36.09 | < 2 × 10⁻¹⁶ | ~26× higher hazard vs. severity 0 |
| **2** | **+3.36** | **28.71** | 21.91 – 37.61 | < 2 × 10⁻¹⁶ | ~29× higher hazard |
| **3** | **+3.28** | **26.59** | 21.59 – 32.76 | < 2 × 10⁻¹⁶ | ~27× higher hazard |
| **4** | **+2.80** | **16.52** | 10.55 – 25.88 | < 2 × 10⁻¹⁶ | ~17× higher hazard |
| **5** | **+3.61** | **36.89** | 27.04 – 50.35 | < 2 × 10⁻¹⁶ | ~37× higher hazard |

### Interpretation
* All severity categories show **very large, highly significant** increases in hazard relative to the reference group.  
* Even the least-severe elevated group (4) has >16-fold risk; severity 5 exceeds 35-fold.  
* A C-index of 0.75 indicates good discrimination (the model correctly orders ~75 % of patient pairs by risk).

## Recommended Next Steps
1. **Check proportional-hazards assumption:** `cox.zph(fit)` and time-dependent Schoenfeld plots.  
2. **Add confounders/modifiers:** age, sex, race, BP, HbA1c, BMI, meds.  
3. **Calibration & visualization:**  
   * Plot observed vs. predicted survival.  
   * Use `ggadjustedcurves()` to display adjusted survival by severity.  
4. **Sensitivity analyses:** alternative severity codings, competing-risk models, stratified analyses.

*Take-home message:* Baseline severity is an exceptionally strong, graded predictor of developing CKD in this diabetic cohort.

## Key Take-Home Messages

* Precise **data wrangling** (start, stop, status) underpins valid survival analysis.
* **Kaplan–Meier** provides descriptive survival; **Cox PH** enables multivariable inference.
* CKD severity shows a strong, graded association with incident CKD in diabetics (HR ≈ 17–37 vs. reference).
* Always test assumptions and balance statistical findings with clinical significance.

## 🎓 Course Wrap-Up & Next Steps  

Congratulations on completing the workshop!  
Everything you coded in the notebook is directly reusable in your own projects.  
Below is a concise **“skills you now own”** checklist followed by **recommended next steps** to keep the momentum going.

---

### ✅ What You Can Now Do on Your Own

1. **Create a “Table 1” for any study cohort**  
   - Summarise categorical variables with counts + percentages.  
   - Summarise continuous variables with mean ± SD *or* median (IQR) after checking distribution.  
   - Run appropriate hypothesis tests: χ² / Fisher’s exact for categorical, t-test / Wilcoxon for continuous.  
   - Assemble a polished, publication-ready table in Markdown/HTML/Word.

2. **Run and interpret Logistic Regression**  
   - Fit univariable and multivariable `glm(..., family = binomial)` models.  
   - Extract odds ratios, 95 % CIs, and p-values with `broom::tidy(exponentiate = TRUE)`.  
   - Automate multi-model summaries with a functional helper (`purrr::imap_dfr`).  
   - Translate odds ratios into plain-language risk statements.

3. **Run and diagnose Linear Regression**  
   - Inspect outcome distributions and decide on transformations if needed.  
   - Build nested models with `lm()` and `update()`.  
   - Compare models using R², adjusted R², AIC, and BIC.  
   - Check assumptions (linearity, homoscedasticity, influence) via diagnostic plots.  
   - Interpret β coefficients in real-world units.

4. **Conduct Survival Analysis**  
   - Reshape “long” condition data into a one-row-per-person survival dataset with `index_date`, `event_date`, `status`, and follow-up time.  
   - Handle right-censoring using study end dates or last-visit dates.  
   - Plot Kaplan–Meier curves with risk tables and log-rank p-values (`survfit`, `ggsurvplot`).  
   - Fit Cox proportional-hazards models (`coxph`), interpret hazard ratios, and test the PH assumption (`cox.zph`).  

---

### 🚀 Suggested Next Steps

| Area | Why It Matters | How to Level Up |
|------|----------------|-----------------|
| **Model Diagnostics & Robustness** | Ensures conclusions hold under scrutiny. | • Learn influence measures (Cook’s D).<br>• Try robust or mixed-effects models. |
| **Variable Selection & Penalisation** | Prevents over-fitting in high-dimensional data. | • Explore stepwise AIC/BIC, LASSO (`glmnet`), or elastic-net. |
| **Effect Modification & Interactions** | Captures different effects across subgroups. | • Add interaction terms (`age*gender`).<br>• Stratified KM curves. |
| **Time-Dependent Covariates** | Many exposures change over time. | • Use `coxph(..., tt = )` or counting-process syntax. |
| **Competing-Risk & Multi-state Models** | When multiple events (death vs. CKD) coexist. | • Look at `cmprsk`, `mstate`, or `flexsurv`. |
| **Reproducible Reporting** | Shareable science saves time. | • Convert notebooks to R Markdown / Quarto.<br>• Use version control (Git). |
| **Power & Sample-Size Calculations** | Plan studies, avoid under-powered results. | • Packages: `pwr`, `powerSurvEpi`. |
| **Advanced Visualisation** | Communicate findings effectively. | • `ggplot2` extensions (`patchwork`, `ggtext`).<br>• Interactive dashboards (Shiny). |

---

> **Action Plan:**  
> 1. Pick one of your own datasets.  
> 2. Replicate each workflow block (Table 1 → logistic → linear → survival as appropriate).  
> 3. Push the code to a GitHub repo and write a short README explaining your findings.  
> 4. Post questions or insights on the course forum—learning accelerates when you teach others.

*Statistical skills compound: keep practicing, iterating, and sharing!*  